# Structural / grouped / netCDF-4 files

Non-gridded files: 1-D series, many-variable tables, hierarchical netCDF-4 groups, and string-heavy metadata. These are about exploring structure rather than spatial plotting/cropping.

In [1]:
%matplotlib inline
from pathlib import Path

from pyramids.netcdf import NetCDF

DATA = Path('../../../../examples/data/netcdf/samples')

> **A note on cropping these files.** None of the files here are 2-D rasters: they hold 1-D station/aircraft time series, 1-D arrays nested in netCDF-4 **groups**, or an L2 satellite swath whose geolocation is 2-D (`LATITUDE(y, x)` / `LONGITUDE(y, x)`). Polygon `crop` is an affine cutline warp that needs a regular lon/lat grid, so it doesn't apply to trajectories, grouped 1-D arrays, or curvilinear swaths — and there is no spatial **group** to clip. The whole-container `nc.crop(...)` shown in `01-gridded-cf` / `02-gridded-coards` / `03-packed-int16` is the file-wide crop; for tabular/station data, filter by coordinate instead.

## `none__1v__1d1.nc`

A minimal single-variable, single-dimension file.

**Open the file and inspect the container**

In [2]:
nc = NetCDF.read_file(DATA / 'none__1v__1d1.nc')
nc

Driver: netCDF/Network Common Data Format
Files: ../../../../examples/data/netcdf/samples/none__1v__1d1.nc
Size is 512, 512
Corner Coordinates:
Upper Left  (    0.0,    0.0)
Lower Left  (    0.0,  512.0)
Upper Right (  512.0,    0.0)
Lower Right (  512.0,  512.0)
Center      (  256.0,  256.0)

**Dimensions and variables**

In [3]:
# get_all_metadata() returns a NetCDFMetadata whose summary lists dimensions and, for each
# variable, its dims / shape / dtype / unit / scale-offset.
meta = nc.get_all_metadata()
print(meta)

NetCDFMetadata
  Driver: netCDF
  Root group: /
  Dimensions (1):
    dim1                 size=10000
  Variables (1):
    var1 dims=('/dim1',) shape=(10000,) dtype=float32
  Global attributes: 


**Global attributes**

In [4]:
nc.global_attributes

{}

**Convert to an xarray Dataset** for a familiar tabular/labelled view

In [5]:
nc.to_xarray()

<xarray.Dataset> Size: 80kB
Dimensions:  (dim1: 10000)
Coordinates:
  * dim1     (dim1) float32 40kB 420.0 197.0 391.5 399.0 ... 155.5 186.5 444.0
Data variables:
    var1     (dim1) float32 40kB 420.0 197.0 391.5 399.0 ... 155.5 186.5 444.0

## `none__11v__1d11.nc`

Eleven 1-D series sharing one dimension (station/time series).

**Open the file and inspect the container**

In [6]:
nc = NetCDF.read_file(DATA / 'none__11v__1d11.nc')
nc

Driver: netCDF/Network Common Data Format
Files: ../../../../examples/data/netcdf/samples/none__11v__1d11.nc
Size is 512, 512
Corner Coordinates:
Upper Left  (    0.0,    0.0)
Lower Left  (    0.0,  512.0)
Upper Right (  512.0,    0.0)
Lower Right (  512.0,  512.0)
Center      (  256.0,  256.0)

**Dimensions and variables**

In [7]:
# get_all_metadata() returns a NetCDFMetadata whose summary lists dimensions and, for each
# variable, its dims / shape / dtype / unit / scale-offset.
meta = nc.get_all_metadata()
print(meta)

NetCDFMetadata
  Driver: netCDF
  Root group: /
  Dimensions (1):
    time                 size=180
  Variables (11):
    Drops dims=('/time',) shape=(180,) dtype=int32 unit='#'
    altitude dims=('/time',) shape=(180,) dtype=float32 unit='km'
    dp dims=('/time',) shape=(180,) dtype=float32 unit='deg_C'
    latitude dims=('/time',) shape=(180,) dtype=float32 unit='degrees_N'
    longitude dims=('/time',) shape=(180,) dtype=float32 unit='degrees_E'
    mr dims=('/time',) shape=(180,) dtype=float32 unit='g/kg'
    pressure dims=('/time',) shape=(180,) dtype=float32 unit='hPa'
    tdry dims=('/time',) shape=(180,) dtype=float32 unit='deg_C'
    time dims=('/time',) shape=(180,) dtype=float64 unit='seconds since 1970-1-1 0:00:00 0:00'
    wdir dims=('/time',) shape=(180,) dtype=float32 unit='degrees'
    ... and 1 more
  Global attributes: history


**Global attributes**

In [8]:
nc.global_attributes

{'history': '$Id: TrackFile.java,v 1.20 2003/05/07 04:53:23 maclean Exp $'}

**Convert to an xarray Dataset** for a familiar tabular/labelled view

In [9]:
nc.to_xarray()

<xarray.Dataset> Size: 9kB
Dimensions:    (time: 180)
Coordinates:
  * time       (time) datetime64[ns] 1kB 2003-07-06T06:30:13 ... 2003-07-06T0...
Data variables:
    altitude   (time) float32 720B 0.103 0.1301 0.1521 ... 11.72 11.75 11.76
    latitude   (time) float32 720B 43.58 43.58 43.58 43.58 ... 42.33 42.33 42.32
    longitude  (time) float32 720B -96.73 -96.73 -96.73 ... -96.27 -96.24 -96.23
    pressure   (time) float32 720B 955.5 955.5 955.5 955.4 ... 217.1 216.6 215.0
    tdry       (time) float32 720B 21.7 21.6 21.6 21.6 ... -47.3 -47.0 -47.8
    dp         (time) float32 720B 14.6 14.8 15.0 15.0 ... -51.4 -52.0 -52.4
    mr         (time) float32 720B 11.06 11.2 11.35 ... 0.1504 0.1404 0.1349
    wspd       (time) float32 720B 11.7 11.1 10.7 11.2 ... 14.9 11.1 13.5 16.3
    wdir       (time) float32 720B 176.9 176.9 176.9 176.9 ... 202.0 201.2 202.3
    Drops      (time) int32 720B 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 1 1 1 1 1 1 1
Attributes:
    history:  $Id: TrackFile.java,v 1.20 2003/05/07 04:53:23 maclean Exp $

## `cf__40v__1d28-2d9-3d3__nc4.nc`

A netCDF-4 file with many variables.

**Open the file and inspect the container**

In [10]:
nc = NetCDF.read_file(DATA / 'cf__40v__1d28-2d9-3d3__nc4.nc')
nc

Driver: netCDF/Network Common Data Format
Files: ../../../../examples/data/netcdf/samples/cf__40v__1d28-2d9-3d3__nc4.nc
Size is 512, 512
Corner Coordinates:
Upper Left  (    0.0,    0.0)
Lower Left  (    0.0,  512.0)
Upper Right (  512.0,    0.0)
Lower Right (  512.0,  512.0)
Center      (  256.0,  256.0)

**Dimensions and variables**

In [11]:
# get_all_metadata() returns a NetCDFMetadata whose summary lists dimensions and, for each
# variable, its dims / shape / dtype / unit / scale-offset.
meta = nc.get_all_metadata()
print(meta)

NetCDFMetadata
  Driver: netCDF
  Root group: /
  Dimensions (6):
    DATETIME             size=2
    PRESSURE             size=18
    independent_2        size=2
    independent_22       size=22
    independent_465      size=465
    independent_7        size=7
  Variables (40):
    ALTITUDE_BOUNDARIES dims=('/DATETIME', '/PRESSURE', '/independent_2') shape=(2, 18, 2) dtype=float32 unit='km'
    APrioriCovarianceMatrix dims=('/DATETIME', '/independent_465') shape=(2, 465) dtype=int16 unit='1'
    AerosolOpticalThickness dims=('/DATETIME',) shape=(2,) dtype=float32 unit='1'
    CloudPressure dims=('/DATETIME',) shape=(2,) dtype=int16 unit='hPa'
    ColumnAmountO3 dims=('/DATETIME',) shape=(2,) dtype=float32 unit='DU'
    CovarianceMatrix dims=('/DATETIME', '/independent_465') shape=(2, 465) dtype=int16
    DATETIME dims=('/DATETIME',) shape=(2,) dtype=float64 unit='days since 2000-01-01'
    DegreesOfFreedomForSignal dims=('/DATETIME',) shape=(2,) dtype=int16 unit='1'
    EffectiveCloud

**Global attributes**

In [12]:
nc.global_attributes

{'Conventions': 'CF-1.6',
 'FILE_NAME': 'OMI-Aura_L2-OMO3PR_2007m1120t1304-o17814_v003-2009m0910t070427.nc',
 'Original_Extension': '.he5',
 'FILE_HISTORY': "omihdf.py OMI-Aura_L2-OMO3PR_2007m1120t1304-o17814_v003-2009m0910t070427.he5\ngtfilter -o /nobackup/users/piters/data/OMI/ominc/filtered/OMI-Aura_L2-OMO3PR_2007m1120t1304-o17814_v003-2009m0910t070427.nc -f 'LATITUDE>-26.9713137991;LATITUDE<54.9713137991;LONGITUDE>-74.0579574717;LONGITUDE<85.5779574717' /nobackup/users/piters/data/OMI/ominc/tmp/OMI-Aura_L2-OMO3PR_2007m1120t1304-o17814_v003-2009m0910t070427.nc\ngtfilter -c 'O3.COLUMN.PARTIAL[DU];O3.COLUMN.PARTIAL_UNCERTAINTY[DU];DATETIME[days since 2000-01-01]' -fiv O3.COLUMN.PARTIAL -o satellite/converted/OMI-Aura_L2-OMO3PR_2007m1120t1304-o17814_v003-2009m0910t070427.nc /nobackup/users/piters/data/OMI/ominc/filtered/OMI-Aura_L2-OMO3PR_2007m1120t1304-o17814_v003-2009m0910t070427.nc\ngtfilter -fca collocation_result_a.csv -o satellite/collocated/OMI-Aura_L2-OMO3PR_2007m1120t1304-o178

**Convert to an xarray Dataset** for a familiar tabular/labelled view

In [13]:
nc.to_xarray()

/home/runner/work/pyramids/pyramids/.pixi/envs/docs/lib/python3.14/site-packages/xarray/namedarray/core.py:261: UserWarning: Duplicate dimension names present: dimensions {'PRESSURE'} appear more than once in dims=('DATETIME', 'PRESSURE', 'PRESSURE'). We do not yet support duplicate dimension names, but we do allow initial construction of the object. We recommend you rename the dims immediately to become distinct, as most xarray functionality is likely to fail silently if you do not. To rename the dimensions you will need to set the ``.dims`` attribute of each variable, ``e.g. var.dims=('x0', 'x1')``.
  self._dims = self._parse_dimensions(dims)


<xarray.Dataset> Size: 7kB
Dimensions:                        (DATETIME: 2, independent_465: 465,
                                    PRESSURE: 18, independent_22: 22,
                                    independent_7: 7, independent_2: 2)
Coordinates:
  * DATETIME                       (DATETIME) datetime64[ns] 16B 2007-11-20T1...
  * PRESSURE                       (PRESSURE) float32 72B 0.3873 ... 836.7
Dimensions without coordinates: independent_465, independent_22, independent_7,
                                independent_2
Data variables: (12/38)
    APrioriCovarianceMatrix        (DATETIME, independent_465) int16 2kB 327 ...
    AerosolOpticalThickness        (DATETIME) float32 8B 0.0 0.0
    O3.COLUMN.PARTIAL_AVK          (DATETIME, PRESSURE, PRESSURE) int16 1kB 4...
    CloudPressure                  (DATETIME) int16 4B 470 497
    ColumnAmountO3                 (DATETIME) float32 8B 299.3 298.0
    CovarianceMatrix               (DATETIME, independent_465) int16 2kB 76 ....
    ...                             ...
    SolarZenithAngle               (DATETIME) int16 4B 7693 7741
    TerrainHeight                  (DATETIME) int16 4B 0 0
    ViewingAzimuthAngle            (DATETIME) int16 4B -8573 -8560
    ViewingZenithAngle             (DATETIME) int16 4B 6698 6698
    INDEX                          (DATETIME) int32 8B 8309 8339
    INDEX.COLLOCATION              (DATETIME) int32 8B 7493 7494
Attributes: (12/128)
    Conventions:                           CF-1.6
    FILE_NAME:                             OMI-Aura_L2-OMO3PR_2007m1120t1304-...
    Original_Extension:                    .he5
    FILE_HISTORY:                          omihdf.py OMI-Aura_L2-OMO3PR_2007m...
    OPF_automaticQualitySuspect:           50
    OPF_LMLangrangeParamScaleFactor:       10.0
    ...                                    ...
    QAPctEclipse:                          0
    QAPctInitializationWarning:            21
    QAPctOptimalEstimationConvergence:     0
    OPF_reflCostFunctionThreshold:         5
    OPF_numStreamLIDORT:                   6
    FILE_INDEX:                            70

## `none__35v__1d35__groups-nc4.nc`

A netCDF-4 file organised into hierarchical groups.

**Open the file and inspect the container**

In [14]:
nc = NetCDF.read_file(DATA / 'none__35v__1d35__groups-nc4.nc')
nc

Driver: netCDF/Network Common Data Format
Files: ../../../../examples/data/netcdf/samples/none__35v__1d35__groups-nc4.nc
Size is 512, 512
Corner Coordinates:
Upper Left  (    0.0,    0.0)
Lower Left  (    0.0,  512.0)
Upper Right (  512.0,    0.0)
Lower Right (  512.0,  512.0)
Center      (  256.0,  256.0)

**Dimensions and variables**

In [15]:
# get_all_metadata() returns a NetCDFMetadata whose summary lists dimensions and, for each
# variable, its dims / shape / dtype / unit / scale-offset.
meta = nc.get_all_metadata()
print(meta)

NetCDFMetadata
  Driver: netCDF
  Root group: /
  Dimensions (7):
    recNum               size=74
    air_press            size=78
    air_press            size=76
    air_press            size=78
    air_press            size=60
    air_press            size=60
    air_press            size=75
  Variables (35):
    UTC_time dims=('/recNum',) shape=(74,) dtype=unknown
    CO dims=('/mozaic_flight_2012030319051051_descent/air_press',) shape=(78,) dtype=float64
    O3 dims=('/mozaic_flight_2012030319051051_descent/air_press',) shape=(78,) dtype=float64
    UTC_time dims=('/mozaic_flight_2012030319051051_descent/air_press',) shape=(78,) dtype=unknown
    air_press dims=('/mozaic_flight_2012030319051051_descent/air_press',) shape=(78,) dtype=float64
    altitude dims=('/mozaic_flight_2012030319051051_descent/air_press',) shape=(78,) dtype=float64
    CO dims=('/mozaic_flight_2012030321335035_descent/air_press',) shape=(76,) dtype=float64
    O3 dims=('/mozaic_flight_2012030321335035_desce

**Global attributes**

In [16]:
nc.global_attributes

{}

**Convert to an xarray Dataset** for a familiar tabular/labelled view

In [17]:
nc.to_xarray()

/tmp/ipykernel_3717/2267532333.py:1: UserWarning: to_xarray() renamed 8 variable(s): ['mozaic_flight_2012030403540535_ascent/air_press -> mozaic_flight_2012030403540535_ascent_air_press', 'mozaic_flight_2012030403540535_ascent/CO -> mozaic_flight_2012030403540535_ascent_CO', 'mozaic_flight_2012030403540535_ascent/O3 -> mozaic_flight_2012030403540535_ascent_O3', 'mozaic_flight_2012030403540535_ascent/altitude -> mozaic_flight_2012030403540535_ascent_altitude', 'mozaic_flight_2012030321335035_descent/CO -> mozaic_flight_2012030321335035_descent_CO', 'mozaic_flight_2012030321335035_descent/O3 -> mozaic_flight_2012030321335035_descent_O3', 'mozaic_flight_2012030321335035_descent/altitude -> mozaic_flight_2012030321335035_descent_altitude', 'mozaic_flight_2012030321335035_descent/UTC_time -> mozaic_flight_2012030321335035_descent_UTC_time']. An xarray Dataset is one flat namespace and netCDF forbids '/' in a name, so a sub-group array's path is flattened with '_', and where that lands on a 

<xarray.Dataset> Size: 21kB
Dimensions:                                          (recNum: 74, air_press: 76)
Coordinates:
  * recNum                                           (recNum) <U19 6kB '2012-...
Dimensions without coordinates: air_press
Data variables:
    UTC_time                                         (recNum) <U19 6kB '2012-...
    mozaic_flight_2012030403540535_ascent_air_press  (recNum) float64 592B 1....
    mozaic_flight_2012030403540535_ascent_CO         (recNum) float64 592B 21...
    mozaic_flight_2012030403540535_ascent_O3         (recNum) float64 592B -9...
    mozaic_flight_2012030403540535_ascent_altitude   (recNum) float64 592B 46...
    mozaic_flight_2012030321335035_descent_CO        (air_press) float64 608B ...
    mozaic_flight_2012030321335035_descent_O3        (air_press) float64 608B ...
    mozaic_flight_2012030321335035_descent_altitude  (air_press) float64 608B ...
    mozaic_flight_2012030321335035_descent_UTC_time  (air_press) <U19 6kB '20...

## `none__111v__1d96-2d13-3d2__str.nc`

A very wide, string-heavy table-like file (111 variables).

**Open the file and inspect the container**

In [18]:
nc = NetCDF.read_file(DATA / 'none__111v__1d96-2d13-3d2__str.nc')
nc

Driver: netCDF/Network Common Data Format
Files: ../../../../examples/data/netcdf/samples/none__111v__1d96-2d13-3d2__str.nc
Size is 512, 512
Corner Coordinates:
Upper Left  (    0.0,    0.0)
Lower Left  (    0.0,  512.0)
Upper Right (  512.0,    0.0)
Lower Right (  512.0,  512.0)
Center      (  256.0,  256.0)

**Dimensions and variables**

In [19]:
# get_all_metadata() returns a NetCDFMetadata whose summary lists dimensions and, for each
# variable, its dims / shape / dtype / unit / scale-offset.
meta = nc.get_all_metadata()
print(meta)

NetCDFMetadata
  Driver: netCDF
  Root group: /
  Dimensions (22):
    ICcheckNameLen       size=72
    ICcheckNum           size=55
    QCcheckNameLen       size=60
    QCcheckNum           size=10
    maxAutoStaLen        size=6
    maxAutoWeaLen        size=12
    maxAutoWeather       size=5
    maxCldTypeLen        size=5
    maxCloudTypes        size=5
    maxDataSrcLen        size=8
    maxRepLen            size=5
    maxSAOLen            size=256
    maxSkyCover          size=5
    maxSkyLen            size=8
    maxSkyMethLen        size=3
    maxStaNamLen         size=5
    maxStaticIds         size=350
    maxWeatherLen        size=40
    maxWeatherNum        size=5
    nInventoryBins       size=24
    recNum               size=178
    totalIdLen           size=6
  Variables (111):
    ICT dims=('/ICcheckNum',) shape=(55,) dtype=unknown
    QCT dims=('/QCcheckNum',) shape=(10,) dtype=unknown
    altimeter dims=('/recNum',) shape=(178,) dtype=float32 unit='1e2 pascals'
    alt

**Global attributes**

In [20]:
nc.global_attributes

{'cdlDate': '20010327',
 'idVariables': 'stationName',
 'timeVariables': 'timeObs',
 'filePeriod': 3600,
 'fileEndOffset': 2640,
 'DD_long_name': 'QC data descriptor model:  QC summary values',
 'DD_reference': 'AWIPS Technique Specification Package (TSP) 88-21-R2',
 'DD_values': 'Z,C,S,V,X,Q,K,k,G, or B',
 'DD_value_Z': 'No QC applied',
 'DD_value_C': 'Passed QC stage 1',
 'DD_value_S': 'Passed QC stages 1 and 2',
 'DD_value_V': 'Passed QC stages 1, 2 and 3',
 'DD_value_X': 'Failed QC stage 1',
 'DD_value_Q': 'Passed QC stage 1, but failed stages 2 or 3 ',
 'DD_value_K': 'Passed QC stages 1, 2, 3, and 4',
 'DD_value_k': 'Passed QC stage 1,2, and 3, failed stage 4 ',
 'DD_value_G': 'Included in accept list',
 'DD_value_B': 'Included in reject list',
 'QCStage_long_name': 'automated QC checks contained in each stage',
 'QCStage_values': '1, 2, 3, or 4',
 'QCStage_value_1': 'Validity and Position Consistency Check',
 'QCStage_value_2': 'Internal, Temporal, and Model Consistency Checks',


**Convert to an xarray Dataset** for a familiar tabular/labelled view

In [21]:
nc.to_xarray()

<xarray.Dataset> Size: 414kB
Dimensions:              (maxStaticIds: 350, recNum: 178, nInventoryBins: 24,
                          QCcheckNum: 10, ICcheckNum: 55, maxSkyLen: 8,
                          maxSkyCover: 5, maxSkyMethLen: 3)
Coordinates:
  * QCcheckNum           (QCcheckNum) <U45 2kB '1- Validity Check' ... ''
  * ICcheckNum           (ICcheckNum) <U72 16kB '1- Sea-Level pressure vs. St...
Dimensions without coordinates: maxStaticIds, recNum, nInventoryBins,
                                maxSkyLen, maxSkyCover, maxSkyMethLen
Data variables: (12/111)
    staticIds            (maxStaticIds) <U3 4kB 'WAF' 'WAH' 'WAJ' ... '' '' ''
    lastRecord           (maxStaticIds) int32 1kB 172 115 72 174 ... -1 -1 -1 -1
    invTime              (recNum) int32 712B 1034088300 ... 1034091840
    prevRecord           (recNum) int32 712B -1 -1 -1 -1 -1 ... -1 124 120 -1
    inventory            (maxStaticIds) int32 1kB 262160 64 16 64 64 ... 0 0 0 0
    isOverflow           (recNum) int32 712B 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
    ...                   ...
    pressChange3HourQCR  (recNum) int32 712B 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
    pressChange3HourQCD  (recNum, QCcheckNum) float32 7kB 3.403e+38 ... 3.403...
    pressChange3HourICA  (recNum) int32 712B 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
    pressChange3HourICR  (recNum) int32 712B 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
    rawSAO               (recNum) <U255 182kB 'WRN SA 1445 AUTO4 M M M 151/12...
    correction           (recNum) int32 712B 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
Attributes: (12/83)
    cdlDate:                  20010327
    idVariables:              stationName
    timeVariables:            timeObs
    filePeriod:               3600
    fileEndOffset:            2640
    DD_long_name:             QC data descriptor model:  QC summary values
    ...                       ...
    ICR_long_name:            IC results Model:  results word definition
    ICR_NoBitsSet:            No IC applied
    ICR_Bit1Set:              Master bit - at least 1 check applied
    ICR_BitiSet:              IC check # applied
    ICR_LeastSignificantBit:  bit1
    ICR_reference:            IC check #'s defined in IC check table